# 🌍 Khởi động Dự án từ GitHub trên Google Colab

Notebook này sẽ clone mã nguồn từ GitHub thẳng vào môi trường máy ảo của Colab (`/content/`), kết nối Google Drive để lưu logs và chép/giải nén Dataset từ Drive của bạn vào máy ảo.

In [ ]:
# 1. Xóa các tệp rác mặc định của Colab (nếu có) để tạo không gian sạch
%cd /content
!rm -rf sample_data

# 2. Clone dự án từ GitHub (Thay link bằng link dự án của bạn nếu cần)
!git clone https://github.com/GnurTinz/DAP_391m repo_tmp

# 3. Đẩy toàn bộ code từ thư mục clone ra đường dẫn chính (/content/)
!mv repo_tmp/* /content/
!mv repo_tmp/.* /content/ 2>/dev/null || true  # Di chuyển cả file ẩn (như .gitignore)
!rm -rf repo_tmp

print("✅ Đã tải và đẩy mã nguồn ra đường dẫn chính:")
!ls -la

In [ ]:
# 4. Cài đặt môi trường (Các thư viện theo requirements)
!pip install torch torchvision pytorch-lightning hydra-core tensorboard matplotlib numpy

## ☁️ Kết nối Google Drive (Lưu Logs & Tải Dataset)

In [ ]:
# 5. Kết nối Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 6. Thiết lập thư mục lưu kết quả tự động (Logs)
# Mọi logs (TensorBoard, Checkpoints, Ảnh sinh ra) sẽ được tự động đổ về Drive của bạn
!mkdir -p /content/drive/MyDrive/palm_logs
!rm -rf /content/logs
!ln -s /content/drive/MyDrive/palm_logs /content/logs
print("✅ Mọi kết quả huấn luyện (trong thư mục logs/) sẽ được tự động lưu về Google Drive!")

### 🗂 Hướng dẫn nạp Dataset từ Google Drive
Bạn có thể nén bộ Dataset của mình (ví dụ `Tongji.zip/rar` hoặc `IITD.zip/rar`), upload lên Google Drive (vào thẳng thư mục gốc `MyDrive`), sau đó chạy đoạn lệnh dưới đây để giải nén trực tiếp vào thư mục `/content/data/` của máy ảo giúp tốc độ đọc dữ liệu cực nhanh.

In [ ]:
# 7. Chép và giải nén Dataset từ Drive vào máy ảo
# Lưu ý: Bỏ comment dòng lệnh tương ứng với đuôi file của bạn (ZIP hoặc RAR)
!mkdir -p /content/data

# ============ DÀNH CHO FILE ZIP ============
# !unzip -q /content/drive/MyDrive/Tongji.zip -d /content/data/Tongji
# !unzip -q /content/drive/MyDrive/IITD.zip -d /content/data/IITD

# ============ DÀNH CHO FILE RAR ============
# !mkdir -p /content/data/Tongji
# !unrar x /content/drive/MyDrive/Tongji.rar /content/data/Tongji/

# !mkdir -p /content/data/IITD
# !unrar x /content/drive/MyDrive/IITD.rar /content/data/IITD/

print("✅ Đã chuẩn bị xong thư mục dữ liệu:")
!ls -la /content/data

## 🚀 Tiến hành Huấn Luyện (Sử dụng hệ thống Hydra)
Hệ thống hiện tại sử dụng Hydra để cấu hình lệnh linh hoạt. Bạn có thể thay đổi dataset (`tongji_session`, `tongji_mixed`, `iitd_hand`, vv) và cấu hình ngay trên câu lệnh.

In [ ]:
# Huấn luyện mô hình với dataset là Tongji (chế độ mixed)
# Lưu ý truyền đúng đường dẫn thư mục dataset nếu nó khác data/Tongji
!python tools/train_lightning.py \
    dataset=tongji_mixed \
    dataset.data_dir=/content/data/Tongji \
    model=unet_mock \
    logging=colab \
    training.epochs=50 \
    training.log_interval=20

In [ ]:
# (Ví dụ khác) Huấn luyện mô hình với dataset là IITD (chế độ chia theo tay trái/phải)
# !python tools/train_lightning.py \
#     dataset=iitd_hand \
#     dataset.data_dir=/content/data/IITD \
#     model=unet_mock \
#     logging=colab \
#     training.epochs=50